# DS2002 · Pandas Core Ops

**Studio — 2026-09-16 · Fall 2026**  
**Class time:** 45 minutes

---

## Building a vendor report

Today you build one deliverable end to end: a per-vendor summary that a game-day manager could act on. Three sources, a join that does not behave, and a report at the end.

The data has problems planted in it. Finding them is part of the work — a report you cannot defend is worth nothing, however good the code looks.

In [5]:
import pandas as pd
from io import StringIO

orders = pd.read_csv(StringIO('''order_id,vendor_id,item,qty,price
1,V-01,Cheeseburger,2,7.50
2,V-10,Foam Finger,1,12.00
3,V-01,Hot Dog,3,4.50
4,V-18,Rain Poncho,5,6.00
5,V-10,UVA T-Shirt,1,24.00
6,V-05,Chicken Tacos,4,6.50
7,V-18,Rain Poncho,8,6.00
8,V-42,Kettle Corn,3,5.00'''))

vendors = pd.read_csv(StringIO('''vendor_id,vendor_name,zone
V-01,Hoos Burgers,A
V-05,Rotunda Tacos,B
V-10,Cav Merch North,A
V-18,Rally Rain Gear,C
V-18,Rally Rain Gear,C'''))

targets = pd.read_csv(StringIO('''zone,revenue_target
A,80
B,25
C,60'''))

print('orders:', orders.shape, '| vendors:', vendors.shape, '| targets:', targets.shape)
orders

orders: (8, 5) | vendors: (5, 3) | targets: (3, 2)


,order_id,vendor_id,item,qty,price
0,1,V-01,Cheeseburger,2,7.5
1,2,V-10,Foam Finger,1,12.0
2,3,V-01,Hot Dog,3,4.5
3,4,V-18,Rain Poncho,5,6.0
4,5,V-10,UVA T-Shirt,1,24.0
5,6,V-05,Chicken Tacos,4,6.5
6,7,V-18,Rain Poncho,8,6.0
7,8,V-42,Kettle Corn,3,5.0


### Worked example — the merge, done carefully

Here is one merge done properly, so the pattern is on the screen before you write anything. Three things happen: record the baseline, merge with `indicator=True`, then compare against the baseline.

In [6]:
baseline_rows = len(orders)
orders['revenue'] = orders['qty'] * orders['price']
baseline_revenue = orders['revenue'].sum()
print(f'before: {baseline_rows} rows, ${baseline_revenue:.2f}')

check = orders.merge(vendors, on='vendor_id', how='left', indicator=True)
print(f'after:  {len(check)} rows, ${check["revenue"].sum():.2f}')
print()
print(check['_merge'].value_counts())

before: 8 rows, $183.50
after:  10 rows, $261.50

_merge
both          9
left_only     1
right_only    0
Name: count, dtype: int64


Eight orders went in and nine came out, and the revenue total moved. Both symptoms point at the same cause, and it is in the `vendors` table, not in the orders.

Find it before you go further — everything downstream inherits this bug.

In [7]:
# Which vendor_id appears more than once in the vendor list?
print(vendors['vendor_id'].value_counts())
print()
print('duplicated vendor rows:', vendors.duplicated().sum())

vendor_id
V-18    2
V-01    1
V-05    1
V-10    1
Name: count, dtype: int64

duplicated vendor rows: 1


### Build 1 — fix the vendor list, then merge

**TODO:** drop the duplicate vendor row, then join it onto `orders` with an indicator. Your merge must come out at **8 rows** with the revenue total unchanged from the baseline. Print both to prove it.

In [8]:
clean_vendors = vendors.drop_duplicates()
joined = orders.merge(clean_vendors, on='vendor_id', how='left', indicator=True)

print(f'Rows after merge: {len(joined)} (expected: {baseline_rows})')
print(f'Revenue after merge: ${joined["revenue"].sum():.2f} (expected: ${baseline_revenue:.2f})')
print('\nMerge indicator counts:')
print(joined['_merge'].value_counts())

Rows after merge: 8 (expected: 8)
Revenue after merge: $183.50 (expected: $183.50)

Merge indicator counts:
_merge
both          7
left_only     1
right_only    0
Name: count, dtype: int64


### Build 2 — handle the vendor nobody has heard of

One order belongs to a vendor that is not on the roster. You have three options, and this is a judgment call:

1. Drop it — clean report, understated revenue.
2. Keep it with a blank name — honest, but it will show up as `NaN` in every chart.
3. Label it `'Unknown vendor'` and keep it in a zone called `'Unassigned'`.

**TODO:** pick one, implement it, and write one sentence saying why. Print how much revenue the decision affects either way.

In [9]:
# Option 3: keep the order, but label it, so the revenue stays on the report.
unmatched = joined[joined['_merge'] == 'left_only']
unmatched_rev = unmatched['revenue'].sum()
kept_total = baseline_revenue
dropped_total = baseline_revenue - unmatched_rev

print('Unmatched order(s):')
print(unmatched[['order_id', 'vendor_id', 'item', 'qty', 'revenue']].to_string(index=False))
print()
print(f'Revenue attached to the unmatched order: ${unmatched_rev:.2f}')
print(f'Share of the day total: {unmatched_rev / baseline_revenue * 100:.1f}%')
print(f'  drop it -> report shows ${dropped_total:.2f}')
print(f'  keep it -> report shows ${kept_total:.2f}')

# Implement the choice: label the stranger instead of dropping it.
report = joined.copy()
report['vendor_name'] = report['vendor_name'].fillna('Unknown vendor')
report['zone'] = report['zone'].fillna('Unassigned')
report = report.drop(columns='_merge')

report_rows = len(report)
report_rev = report['revenue'].sum()
print()
print(f'Rows kept: {report_rows} | revenue on the report: ${report_rev:.2f}')
print(report[['order_id', 'vendor_id', 'vendor_name', 'zone', 'revenue']].to_string(index=False))

Unmatched order(s):
 order_id vendor_id        item  qty  revenue
        8      V-42 Kettle Corn    3     15.0

Revenue attached to the unmatched order: $15.00
Share of the day total: 8.2%
  drop it -> report shows $168.50
  keep it -> report shows $183.50

Rows kept: 8 | revenue on the report: $183.50
 order_id vendor_id     vendor_name       zone  revenue
        1      V-01    Hoos Burgers          A     15.0
        2      V-10 Cav Merch North          A     12.0
        3      V-01    Hoos Burgers          A     13.5
        4      V-18 Rally Rain Gear          C     30.0
        5      V-10 Cav Merch North          A     24.0
        6      V-05   Rotunda Tacos          B     26.0
        7      V-18 Rally Rain Gear          C     48.0
        8      V-42  Unknown vendor Unassigned     15.0


**My decision, and why:** I kept the V-42 order and labeled it `Unknown vendor` in a zone called `Unassigned`.
It is $15.00 that actually came through the till — 8.2% of the day — so dropping it would understate revenue,
and a blank name would quietly vanish into every chart. Labeling it keeps the total honest and leaves the roster
gap visible, which is the thing someone should go fix before the next game.

### Build 3 — the per-vendor summary

**TODO:** one row per vendor, with:

- `orders` — how many orders
- `units` — total quantity
- `revenue` — total revenue
- `avg_ticket` — average revenue per order, rounded to 2 decimals

Sorted by revenue, highest first. Use `.agg()` with named outputs so the columns come out with the names above.

In [10]:
summary = (
    report.groupby(['vendor_id', 'vendor_name'], as_index=False)
          .agg(orders=('order_id', 'nunique'),
               units=('qty', 'sum'),
               revenue=('revenue', 'sum'))
)
summary['avg_ticket'] = (summary['revenue'] / summary['orders']).round(2)
summary = summary.sort_values('revenue', ascending=False).reset_index(drop=True)

summary_rev = summary['revenue'].sum()
print(f'Check: rows {len(summary)} | revenue ${summary_rev:.2f} (baseline ${baseline_revenue:.2f})')
summary

Check: rows 5 | revenue $183.50 (baseline $183.50)


,vendor_id,vendor_name,orders,units,revenue,avg_ticket
0,V-18,Rally Rain Gear,2,13,78.0,39.00
1,V-10,Cav Merch North,2,2,36.0,18.00
2,V-01,Hoos Burgers,2,5,28.5,14.25
3,V-05,Rotunda Tacos,1,4,26.0,26.00
4,V-42,Unknown vendor,1,3,15.0,15.00


### Build 4 — did each zone hit its target?

**TODO:** total revenue by zone, join `targets` on, and add a `hit_target` boolean column. Then print a one-line sentence for each zone that a manager could read.

In [11]:
by_zone = (
    report.groupby('zone', as_index=False)
          .agg(revenue=('revenue', 'sum'))
          .merge(targets, on='zone', how='left')
)
by_zone['hit_target'] = by_zone['revenue'] >= by_zone['revenue_target']
by_zone = by_zone.sort_values('zone').reset_index(drop=True)
print(by_zone.to_string(index=False))
print()

for row in by_zone.itertuples():
    if pd.isna(row.revenue_target):
        print(f'Zone {row.zone}: ${row.revenue:.2f} in revenue and no target on file - that is the unknown vendor, worth chasing down.')
    elif row.hit_target:
        print(f'Zone {row.zone}: hit target - ${row.revenue:.2f} against ${row.revenue_target:.2f}, ${row.revenue - row.revenue_target:.2f} over.')
    else:
        print(f'Zone {row.zone}: missed target - ${row.revenue:.2f} against ${row.revenue_target:.2f}, ${row.revenue_target - row.revenue:.2f} short.')

      zone  revenue  revenue_target  hit_target
         A     64.5            80.0       False
         B     26.0            25.0        True
         C     78.0            60.0        True
Unassigned     15.0             NaN       False

Zone A: missed target - $64.50 against $80.00, $15.50 short.
Zone B: hit target - $26.00 against $25.00, $1.00 over.
Zone C: hit target - $78.00 against $60.00, $18.00 over.
Zone Unassigned: $15.00 in revenue and no target on file - that is the unknown vendor, worth chasing down.


### Build 5 — the one number that matters

**TODO:** rain gear is the thing we can actually act on. Print total poncho units sold and what share of overall revenue they represent, as a percentage rounded to one decimal.

In [12]:
ponchos = report[report['item'].str.contains('Poncho', case=False, na=False)]
poncho_units = int(ponchos['qty'].sum())
poncho_rev = ponchos['revenue'].sum()
total_rev = report['revenue'].sum()
poncho_share = round(poncho_rev / total_rev * 100, 1)

print(f'Poncho units sold: {poncho_units}')
print(f'Poncho revenue: ${poncho_rev:.2f} of ${total_rev:.2f} total')
print(f'Share of overall revenue: {poncho_share}%')

Poncho units sold: 13
Poncho revenue: $78.00 of $183.50 total
Share of overall revenue: 42.5%


---

## Checkpoint (participation)

Report your row count and revenue total after the merge, and what you decided to do with the unknown vendor.

Work the last few minutes in groups of 4–5, then fill in the cell below **yourself**. Paste the printed output (or a screenshot of it) into this week's **Studio Checkpoint** in Canvas by **Wednesday 11:59pm ET**. One submission per person, not per group.

In [13]:
# Checkpoint
rows_after_merge = len(joined)                            # 8
revenue_after_merge = round(joined['revenue'].sum(), 2)   # matches the baseline
unknown_vendor_call = ('Kept V-42 and labeled it Unknown vendor in zone Unassigned - it is '
                       '$15.00 of real revenue (8.2% of the day), so dropping it would '
                       'understate the total and hide a roster gap worth fixing.')

print('rows after merge:', rows_after_merge)
print('revenue after merge:', revenue_after_merge)
print('unknown vendor:', unknown_vendor_call)

rows after merge: 8
revenue after merge: 183.5
unknown vendor: Kept V-42 and labeled it Unknown vendor in zone Unassigned - it is $15.00 of real revenue (8.2% of the day), so dropping it would understate the total and hide a roster gap worth fixing.
